# E053 — Hard 2-second prefix truncation

Test whether discarding the first 2 s of every wav improves EER vs E052 baseline.
Truncation applied at load time for both train and val.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import copy

import librosa
import numpy as np
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
print(f'{len(manifest)} samples loaded')

222 samples loaded


In [2]:
UBM_COMPONENTS, MAP_R = 32, 16.0
TRUNCATE_S = 2.0  # seconds to discard from the start

def _find_wav(stem, data_dir):
    for sf in ('target_train','target_dev','non_target_train','non_target_dev'):
        p = data_dir / sf / (stem + '.wav')
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _truncate_prefix(y, sr, seconds=TRUNCATE_S):
    """Discard first `seconds` of audio. No-op if the clip is shorter."""
    cut = int(seconds * sr)
    return y[cut:] if len(y) > cut else y

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            log_H = -np.log(np.abs(np.fft.rfft(a, n=512)) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta  = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)  # CMN
    return feat

def _train_ubm(X):
    return GaussianMixture(n_components=UBM_COMPONENTS,
                           covariance_type='tied', max_iter=200,
                           random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:,None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:,None]*mu_hat + (1-alpha[:,None])*ubm.means_
    return adapted

def _aug_codec(y, sr):
    y_down = librosa.resample(y, orig_sr=sr, target_sr=8000)
    return librosa.resample(y_down, orig_sr=8000, target_sr=sr)

def _score_tta(y_orig, sr, adapted, ubm):
    """Speed TTA: average LLR over 0.9×/1.0×/1.1× (E042)."""
    views = [y_orig,
             librosa.effects.time_stretch(y_orig, rate=0.9),
             librosa.effects.time_stretch(y_orig, rate=1.1)]
    scores = []
    for v in views:
        f = _extract_lpcc(v, sr)
        if len(f) == 0:
            continue
        scores.append(float((adapted.score_samples(f) - ubm.score_samples(f)).mean()))
    return float(np.mean(scores)) if scores else 0.0

print('Helpers ready')

Helpers ready


In [3]:
def _build_frames(df, data_dir, seed, truncate=False):
    """Build frame matrix for UBM/MAP training.
    truncate=True: discard first 2 s before feature extraction.
    Always includes pitch aug + codec aug (E052 setup).
    """
    rng = np.random.default_rng(seed)
    X_list, y_list = [], []
    for _, row in df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row['stem'], data_dir), sr=None, mono=True)
        if truncate:
            y_wav = _truncate_prefix(y_wav, sr)
        wavs = [
            y_wav,
            librosa.effects.pitch_shift(y_wav, sr=sr,
                n_steps=float(rng.choice([-2,-1,1,2]))),
            _aug_codec(y_wav, sr),
        ]
        for y_aug in wavs:
            f = _extract_lpcc(y_aug, sr)
            if len(f) == 0:
                continue
            X_list.append(f)
            y_list.extend([row['label']] * len(f))
    return np.vstack(X_list), np.array(y_list)

def _train_e052(df, data_dir, seed):
    """E052 baseline: tied cov + pitch aug + codec aug + speed TTA."""
    X, y = _build_frames(df, data_dir, seed, truncate=False)
    ubm = _train_ubm(X[y==0])
    return ubm, _map_adapt(ubm, X[y==1])

def _train_e052_2s(df, data_dir, seed):
    """E053: same as E052 but discard first 2 s of every wav before anything."""
    X, y = _build_frames(df, data_dir, seed, truncate=True)
    ubm = _train_ubm(X[y==0])
    return ubm, _map_adapt(ubm, X[y==1])

print('Model functions ready')

Model functions ready


In [4]:
MODELS = {
    'E052 (baseline)': (_train_e052,   False),
    'E052 + 2s cut':   (_train_e052_2s, True),
}
CONDS = ['clean', 'codec']

results = {name: {c: [] for c in CONDS} for name in MODELS}
dcf_res = {name: {c: [] for c in CONDS} for name in MODELS}

for model_name, (train_fn, truncate_val) in MODELS.items():
    print(f'\n=== {model_name} ===')
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        print(f'  fold {fold_id}...', end=' ', flush=True)
        ubm, adapted = train_fn(manifest.loc[train_idx], DATA, SEED + fold_id)
        val_df = manifest.loc[val_idx]

        for cond in CONDS:
            scores, labels = [], []
            for _, row in val_df.iterrows():
                y_orig, sr = librosa.load(_find_wav(row['stem'], DATA), sr=None, mono=True)
                if truncate_val:
                    y_orig = _truncate_prefix(y_orig, sr)
                y_s = _aug_codec(y_orig, sr) if cond == 'codec' else y_orig
                scores.append(_score_tta(y_s, sr, adapted, ubm))
                labels.append(row['label'])
            scores = np.array(scores); labels = np.array(labels)
            eer, _ = compute_eer(scores[labels==1], scores[labels==0])
            dcf, _ = compute_min_dcf(scores[labels==1], scores[labels==0])
            results[model_name][cond].append(eer * 100)
            dcf_res[model_name][cond].append(dcf)
        print('done')

print('\nExperiment complete')


=== E052 (baseline) ===
  fold 0... done
  fold 1... done
  fold 2... done

=== E052 + 2s cut ===
  fold 0... done
  fold 1... done
  fold 2... done

Experiment complete


In [5]:
print('=== E053 RESULTS ===\n')
print(f'{"Config":<22} {"Clean EER":>12} {"Codec EER":>13} {"Delta clean":>13} {"Clean DCF":>11} {"Codec DCF":>11}')
print('-' * 87)

baseline_clean = np.mean(results['E052 (baseline)']['clean'])
for model_name in MODELS:
    clean = np.array(results[model_name]['clean'])
    codec = np.array(results[model_name]['codec'])
    cdcf  = np.array(dcf_res[model_name]['clean'])
    kodcf = np.array(dcf_res[model_name]['codec'])
    delta = np.mean(clean) - baseline_clean
    sign  = '+' if delta >= 0 else ''
    print(f'{model_name:<22}'
          f' {np.mean(clean):>6.2f}±{np.std(clean):.2f}%'
          f' {np.mean(codec):>7.2f}±{np.std(codec):.2f}%'
          f' {sign}{delta:>+8.2f}pp'
          f' {np.mean(cdcf):>10.4f}'
          f' {np.mean(kodcf):>10.4f}')

print('\nPer-fold breakdown:')
for model_name in MODELS:
    print(f'  {model_name}:')
    for cond in CONDS:
        folds = results[model_name][cond]
        print(f'    {cond:>6}: ' + '  '.join(f'F{i}={v:.2f}%' for i,v in enumerate(folds)))

print('\n--- VERDICT ---')
delta_clean = np.mean(results['E052 + 2s cut']['clean']) - baseline_clean
verdict = 'ADOPT' if delta_clean <= 0.1 else 'REJECT'
print(f'2s truncation: {verdict} (delta clean = {delta_clean:+.2f}pp, threshold = +0.1pp)')

=== E053 RESULTS ===

Config                    Clean EER     Codec EER   Delta clean   Clean DCF   Codec DCF
---------------------------------------------------------------------------------------
E052 (baseline)          0.46±0.65%    3.33±4.14% +   +0.00pp     0.0093     0.0333
E052 + 2s cut            0.83±0.68%    2.78±1.96% +   +0.37pp     0.0167     0.0556

Per-fold breakdown:
  E052 (baseline):
     clean: F0=1.39%  F1=0.00%  F2=0.00%
     codec: F0=9.17%  F1=0.83%  F2=0.00%
  E052 + 2s cut:
     clean: F0=0.00%  F1=1.67%  F2=0.83%
     codec: F0=4.17%  F1=4.17%  F2=0.00%

--- VERDICT ---
2s truncation: REJECT (delta clean = +0.37pp, threshold = +0.1pp)
